# Chapter 02: Attention mechanisms
## 2.1 Self-attention

For those models such as RNN or encoder-decoder, they may not perform well when transforming one sequence to another sequence. Take encoder-decoder model as an example: for input sequence $\{x_1,\cdots,x_n\}$, the model calculates a *context vector* $\boldsymbol{c}$ and use it to predict the output sequence $\{y_1,\cdots,y_m\}$ step by step. When the input sequence becomes longer, the context $\boldsymbol{c}$ may not carry enough information for prediction, causing some performance problems.

One possible solution is to **let the model look back to the whole input sequence at anytime**. For a query $q_i=x_i$, we take the dot product with each input element $x_j$ to get several **attention scores** as follows
$$\omega_{ij}=x_jq_i^T$$

We expect that the sum of these attention scores is $1$, so we use softmax to normalize
$$a_{ij}=\mathrm{softmax}(\omega_{ij})$$

In a sense, $a_{ij}$ carries information between the similarity between $q_i=x_i$ and $x_j$, so we may do a weighted sum among all input elements to get a *context vector* for query $q_i=x_i$
$$z_i=\sum_{j=1}^n a_{ij}x_j$$

This is the basic idea about self-attention: calculating the **similarity** between different input elements to capture their relevance relationships. This could solve the problem that an encoder-decoder model faces, but the current description is very simple and naive.

Generally, if we have an input matrix
$$X=[x_1,x_2,\cdots,x_n]^T$$

The following matrix multiplication calculates the attention score matrix
$$\Omega=XX^T,\quad \Omega_{ij}=\Omega_{ji}=\omega_{ij}=x_jx_i^T=x_jq_i^T$$

See the example as follows.

In [1]:
import torch

# input: "Your journey starts with one step"
# output_dim=3
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x_1)
   [0.55, 0.87, 0.66], # journey  (x_2)
   [0.57, 0.85, 0.64], # starts   (x_3)
   [0.22, 0.58, 0.33], # with     (x_4)
   [0.77, 0.25, 0.10], # one      (x_5)
   [0.05, 0.80, 0.55]] # step     (x_6)
)

# use matrix multiplication for effciency
attn_scores = inputs @ inputs.T
print(attn_scores)

# use softmax to normalize each row
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)


tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


Using each input element $x_i$ as a query is simple but not useful. Now we propose a trainable way to improve its performance. The basic idea is using **linear projection with trainable weights** to get query $q$, key $k$ and value $v$.

With trainable weights and multiple variables, we could expect the model to *learn how to produce good context vector* to better capture the relationships among different input elements. Of course, this process requires epochs of training.

For input element $x_i$, we introduce three matrices $W_q,W_k,W_v$ and define
$$q_i=x_iW_q,\quad k_i=x_iW_k,\quad v_i=x_iW_v$$

Note that the dimension of the embedded input $x_i$ and that of the query $q_i$ can be different.

In [2]:
# take q_2 as example
x_2 = inputs[1]
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

# initialize the three weight matrices
torch.manual_seed(325)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

tensor([0.5785, 1.0935], grad_fn=<SqueezeBackward4>)


We can use the input matrix $X$ instead of a single input, as shown below.
$$Q=XW_q,\quad K=XW_k,\quad V=XW_v$$

In [3]:
queries = inputs @ W_query
keys = inputs @ W_key 
values = inputs @ W_value

print("querys.shape:", queries.shape)
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

querys.shape: torch.Size([6, 2])
keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


To get the attention weight matrix, we use **scaled-dot product attention** as
$$Z=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

where $d_k$ is the embedding dimension of $W_q,W_k,W_v$, used to prevent the dot products from exploding.

In [4]:
import torch.nn as nn

# self-attention mechanism class
class SelfAttention_v1(nn.Module):

    # initialize the three weight matrices
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # calculate keys, queries and values
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        # scaled-dot product attention
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(325)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))


tensor([[1.0574, 0.6080],
        [1.0940, 0.6253],
        [1.0937, 0.6252],
        [1.0583, 0.6052],
        [1.0653, 0.6107],
        [1.0645, 0.6079]], grad_fn=<MmBackward0>)


The calculations are actually matrix multiplications, and we can use `nn.Linear` from `torch` to gain better performance. This could also provide better initialization scheme.

In [5]:
class SelfAttention_v2(nn.Module):

    # initialize the three weight matrices as linear layers (bias considered)
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(325)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[0.1363, 0.0146],
        [0.1362, 0.0130],
        [0.1361, 0.0127],
        [0.1356, 0.0095],
        [0.1343, 0.0043],
        [0.1363, 0.0131]], grad_fn=<MmBackward0>)


## 2.2 Masked self-attention

One principle for self-attention mechanisms is that **for any given input, the model should never be able to utilize future tokens**; that is, we must mask out the future tokens $x_{i+1},\cdots,x_n$ when calculating the current context vector $z_i$ that belongs to $x_i$.

To mask out future tokens, we can use `torch.tril` to create a mask matrix with elements below the main diagonal set to $1$ and above the main diagonal set to $0$.

In the case above, run the following code to create a mask tensor. Then, we just simply multiply the mask tensor with attention matrix **by elements** to get the masked self-attention matrix.
- To make sure that each row sums to 1, we should use softmax after the mask.

In [6]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

# use element-product
masked_simple = attn_weights*mask_simple
print(masked_simple)

# use softmax to normalize
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])
tensor([[0.2098, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1385, 0.2379, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1390, 0.2369, 0.2326, 0.0000, 0.0000, 0.0000],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.0000, 0.0000],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.0000],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3680, 0.6320, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2284, 0.3893, 0.3822, 0.0000, 0.0000, 0.0000],
        [0.2046, 0.2956, 0.2915, 0.2084, 0.0000, 0.0000],
        [0.1753, 0.2250, 0.2269, 0.1570, 0.2158, 0.0000],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


Using `inf` instead of $0$ achieves a more efficient approach, as shown below.

In [7]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4056, 0.5944, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2566, 0.3741, 0.3693, 0.0000, 0.0000, 0.0000],
        [0.2176, 0.2823, 0.2796, 0.2205, 0.0000, 0.0000],
        [0.1826, 0.2178, 0.2191, 0.1689, 0.2115, 0.0000],
        [0.1473, 0.2033, 0.1996, 0.1500, 0.1160, 0.1839]])


## 2.3 Dropout

Using **dropout** to randomly drop some elements could possibly reduce overfitting during training. When we set a dropout rate of $p\in(0,1)$, a random batch of $p$ percent of elements will be set to $0$, while the other values will be scaled by a factor of $1/(1-p)$. 

In [8]:
torch.manual_seed(325)
# here we set p=0.5
dropout = torch.nn.Dropout(0.5)
example = torch.ones(context_length, context_length)

# the following two output may show different dropout results due to randomness
print(dropout(example))
print(dropout(attn_weights))

tensor([[2., 2., 2., 0., 0., 0.],
        [0., 0., 0., 0., 2., 0.],
        [2., 0., 0., 2., 2., 2.],
        [2., 2., 2., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [0., 0., 2., 2., 0., 2.]])
tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8112, 1.1888, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5132, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4352, 0.0000, 0.0000, 0.4409, 0.0000, 0.0000],
        [0.3652, 0.4357, 0.4383, 0.3378, 0.0000, 0.0000],
        [0.0000, 0.4065, 0.3991, 0.2999, 0.2320, 0.0000]])


## 2.4 Causal self-attention

Now, we can come up with a compact **causal self-attention** class using `torch`.

In [9]:
# we use the example in 2.1 and duplicate it to become a 2-input batch
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # a batch with 2 inputs, each input has 6 tokens with 3 embedding dimension

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # use register_buffer to differ from nn.Parameters()
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape 
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) 
        # in-place operation using _, in order to turn those marked True(1 in mask tensor) to -inf
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], # to prevent num_tokens < context_length
                                -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

torch.Size([2, 6, 3])


In [10]:
torch.manual_seed(325)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.2)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.0226, -0.4899],
         [ 0.1174, -0.1872],
         [ 0.0773, -0.1237],
         [ 0.1195, -0.0703],
         [ 0.1258, -0.0318],
         [ 0.1298,  0.0014]],

        [[-0.0226, -0.4899],
         [ 0.1174, -0.1872],
         [ 0.1643, -0.0917],
         [ 0.0318, -0.0792],
         [ 0.1258, -0.0318],
         [ 0.1450, -0.0098]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


## 2.5 Multi-head attention

What we've talked about before is a general **single-head attention layer**. Now we could simply stack multiple single-head attention modules to obtain a multi-head attention layer.
- This is equivalent to running several single-head attention modules in parallel.
- This allows the model to learn different information.
- We **concat** these outputs before a linear layer, which makes sure the output dimension is right as always.

For one single-head attention output with a shape of `(b,t,d)`, the multi-head attention gives an output with a shape of `(b,t,d*h)`, where `h` represents the number of heads.

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [12]:
torch.manual_seed(325)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.2, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.0226, -0.4899,  0.3452, -0.4095],
         [ 0.1284,  0.0496,  0.5469, -0.3877],
         [ 0.1715,  0.0647,  0.5942, -0.3772],
         [ 0.0951, -0.0559,  0.5431, -0.3352],
         [ 0.1846, -0.0263,  0.4231, -0.2609],
         [ 0.1704,  0.0163,  0.2293, -0.1025]],

        [[-0.0226, -0.4899,  0.3452, -0.4095],
         [ 0.1174, -0.1872,  0.5469, -0.3877],
         [ 0.1643, -0.0917,  0.5042, -0.2704],
         [ 0.0951, -0.0559,  0.5431, -0.3352],
         [ 0.1891,  0.0726,  0.4709, -0.2671],
         [ 0.1704,  0.0163,  0.3678, -0.2121]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])
